In [4]:

import pandas as pd
import plotly.graph_objects as go

ANNI = [2020, 2021, 2022, 2023, 2024]
COL_NUOVI = "Nuovi Utenti"
COL_GIA_IN_CARICO = "Utenti rientrati o già in carico"
COL_TOTALE = "Totale"
COL_REGIONE = "Descrizione Regione"

righe_nazionali = []

for anno in ANNI:
    file_anno = f"serd_{anno}.csv"
    df = pd.read_csv(file_anno, sep=";", encoding="latin1")
    df.columns = [c.strip() for c in df.columns]  # pulisce spazi nei nomi colonna
    df[COL_REGIONE] = df[COL_REGIONE].astype(str).str.strip()

    # alcuni file (es. 2022) contengono una riga di riepilogo "TOTALE" che
    # andrebbe altrimenti sommata due volte insieme ai dati regionali: va esclusa
    df = df[~df[COL_REGIONE].str.upper().str.contains("TOTALE", na=False)]

    for col in [COL_NUOVI, COL_GIA_IN_CARICO, COL_TOTALE]:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    somma = df[[COL_NUOVI, COL_GIA_IN_CARICO, COL_TOTALE]].sum()
    righe_nazionali.append({
        "anno": anno,
        "Nuovi utenti": somma[COL_NUOVI],
        "Già in carico": somma[COL_GIA_IN_CARICO],
        "Totale": somma[COL_TOTALE],
    })

df_naz = pd.DataFrame(righe_nazionali)
ultimo_anno = df_naz["anno"].max()
primo_anno = df_naz["anno"].min()


# 2. COLORI
color_map = {
    "Nuovi utenti": "#5d7ea8",     # blu desaturato
    "Già in carico": "#8c3b45",    # bordeaux desaturato
}

# ---------------------------------------------------------------------------
# 3. GRAFICO - doppio asse Y: le due serie hanno ordini di grandezza molto diversi (~15-17mila contro ~108-115mila)
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=df_naz["anno"], y=df_naz["Già in carico"],
    name="Già in carico", mode="lines+markers",
    line=dict(color=color_map["Già in carico"], width=3),
    marker=dict(size=7),
    yaxis="y1",
))

fig.add_trace(go.Scatter(
    x=df_naz["anno"], y=df_naz["Nuovi utenti"],
    name="Nuovi utenti", mode="lines+markers",
    line=dict(color=color_map["Nuovi utenti"], width=3),
    marker=dict(size=7),
    yaxis="y2",
))

# etichette dirette a fine linea al posto della legenda
# Etichetta "Già in carico"
fig.add_annotation(
    x=ultimo_anno - 0.05,
    y=df_naz.loc[df_naz["anno"] == ultimo_anno, "Già in carico"].values[0],
    text="Già in carico",
    showarrow=False,
    xanchor="right",
    yanchor="top",
    yshift=-8,
    font=dict(color=color_map["Già in carico"], size=13),
    yref="y1",
)

# Etichetta "Nuovi utenti"
fig.add_annotation(
    x=ultimo_anno - 0.05,
    y=df_naz.loc[df_naz["anno"] == ultimo_anno, "Nuovi utenti"].values[0],
    text="Nuovi utenti",
    showarrow=False,
    xanchor="right",
    yanchor="top",
    yshift=-8,
    font=dict(color=color_map["Nuovi utenti"], size=13),
    yref="y2",
)

fig.update_layout(
    xaxis=dict(
    title="Anno",
    tickmode="array",
    tickvals=ANNI,
    range=[2019.8, 2024.2],
),
    yaxis=dict(title="n° utenti già nel SerD in anni precedenti", color=color_map["Già in carico"]), # Già in carico (n. utenti)
    yaxis2=dict(
        title="utenti in trattamento per la 1° volta nell'anno", color=color_map["Nuovi utenti"], #  Nuovi utenti (n. utenti)
        overlaying="y", side="right",
    ),
    template="simple_white",
    showlegend=False,
    margin=dict(b=90),
)

fig.add_annotation(
    text=(""),
    xref="paper", yref="paper",
    x=0, y=-0.30,
    showarrow=False,
    font=dict(size=10, color="#888888"),
    align="left",
)

fig.show()